# Evasive AI Lab — Phase 8
## EvasiveBench v0.1

**What is EvasiveBench?**

A single reproducible benchmark that runs all attack classes from Phases 6 and 7
against a standard target model and produces one unified report card.

Anyone can clone the repo, run this notebook, and get comparable results.
That is what makes it a benchmark rather than just an experiment.

**Attacks covered:**

| Attack | NIST ID | Phase |
|---|---|---|
| Membership Inference — Loss-Based | NISTAML.033 | 6 |
| Membership Inference — Shadow Model | NISTAML.033 | 6 |
| Model Extraction — Logistic Substitute | NISTAML.031 | 7 |
| Model Extraction — MLP Substitute | NISTAML.031 | 7 |

**No GPU needed. No API keys. Runs in ~5 minutes.**

---

In [ ]:
# Cell 1: Environment Setup
!pip install -q scikit-learn numpy

import numpy as np
from sklearn.datasets import load_wine, load_iris
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score
import warnings, json
from datetime import datetime, timezone, date
warnings.filterwarnings("ignore")
print("Environment ready.")

In [ ]:
# Cell 2: Configuration
BENCH_VERSION = "EvasiveBench v0.1"
PHASE         = "Phase 8 - EvasiveBench"
REPO          = "github.com/Aswinbalaji14/evasive-lab"

# All attack classes covered by this benchmark
ATTACKS = [
    "membership_inference_loss_based",
    "membership_inference_shadow",
    "model_extraction_logistic",
    "model_extraction_mlp",
]

print(f"Benchmark : {BENCH_VERSION}")
print(f"Repo      : {REPO}")
print(f"Attacks   : {len(ATTACKS)}")
print()
print("This benchmark runs all attack classes from Phases 6 and 7")
print("against a standard target model and produces a single report card.")

In [ ]:
# Cell 3: Build Target Model
# Same setup used in Phases 6 and 7 for consistency

from sklearn.datasets import load_wine
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

data   = load_wine()
X, y   = data.data, data.target
scaler = StandardScaler()
X      = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

target = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),
    max_iter=1000,
    random_state=42
)
target.fit(X_train, y_train)

train_acc = accuracy_score(y_train, target.predict(X_train))
test_acc  = accuracy_score(y_test,  target.predict(X_test))
gap       = train_acc - test_acc

print("TARGET MODEL")
print(f"Architecture : MLP(128, 64, 32)")
print(f"Dataset      : Wine ({len(X)} samples, {X.shape[1]} features)")
print(f"Train acc    : {train_acc:.4f}")
print(f"Test acc     : {test_acc:.4f}")
print(f"Gen gap      : {gap:.4f}")

In [ ]:
# Cell 4: Attack 1 — Membership Inference (Loss-Based)
# Yeom et al. 2018 — NISTAML.033
import numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score

def get_loss(model, X, y):
    probs = model.predict_proba(X)
    return np.array([-np.log(np.clip(probs[i][y[i]], 1e-10, 1.0)) for i in range(len(y))])

member_losses    = get_loss(target, X_train, y_train)
nonmember_losses = get_loss(target, X_test,  y_test)
all_losses = np.concatenate([member_losses, nonmember_losses])
all_labels = np.concatenate([np.ones(len(X_train)), np.zeros(len(X_test))])

threshold   = np.mean(all_losses)
predictions = (all_losses < threshold).astype(int)

mi_loss_asr = accuracy_score(all_labels, predictions)
mi_loss_auc = roc_auc_score(all_labels, -all_losses)

print(f"MI Loss-Based  ASR: {mi_loss_asr*100:.2f}%  AUC: {mi_loss_auc:.4f}")

In [ ]:
# Cell 5: Attack 2 — Membership Inference (Shadow Models)
# Shokri et al. 2017 — NISTAML.033
from sklearn.linear_model import LogisticRegression
import numpy as np

N_SHADOW = 50
shadow_features, shadow_labels = [], []

for i in range(N_SHADOW):
    idx   = np.random.permutation(len(X))
    split = len(X) // 2
    sh = MLPClassifier(hidden_layer_sizes=(64,32), max_iter=500, random_state=i, alpha=0.0001)
    sh.fit(X[idx[:split]], y[idx[:split]])
    shadow_features.append(sh.predict_proba(X[idx[:split]]))
    shadow_labels.append(np.ones(len(idx[:split])))
    shadow_features.append(sh.predict_proba(X[idx[split:]]))
    shadow_labels.append(np.zeros(len(idx[split:])))

meta = LogisticRegression(max_iter=1000, random_state=42)
meta.fit(np.vstack(shadow_features), np.concatenate(shadow_labels))

all_conf = np.vstack([target.predict_proba(X_train), target.predict_proba(X_test)])
all_lbl  = np.concatenate([np.ones(len(X_train)), np.zeros(len(X_test))])

mi_shadow_asr = accuracy_score(all_lbl, meta.predict(all_conf))
mi_shadow_auc = roc_auc_score(all_lbl, meta.predict_proba(all_conf)[:,1])

print(f"MI Shadow      ASR: {mi_shadow_asr*100:.2f}%  AUC: {mi_shadow_auc:.4f}")

In [ ]:
# Cell 6: Attack 3 & 4 — Model Extraction
# Tramer et al. 2016 — NISTAML.031
import numpy as np

query_count = 0
def oracle(Xq):
    global query_count
    query_count += len(Xq)
    return target.predict(Xq)

np.random.seed(99)
X_pool = np.random.uniform(X.min(axis=0), X.max(axis=0), size=(500, X.shape[1]))
X_q    = X_pool[:100]
y_q    = oracle(X_q)

# Logistic regression substitute
lr_sub = LogisticRegression(max_iter=500, random_state=42)
lr_sub.fit(X_q, y_q)
me_lr_fidelity = accuracy_score(oracle(X_test), lr_sub.predict(X_test))
me_lr_acc      = accuracy_score(y_test, lr_sub.predict(X_test))

# MLP substitute
mlp_sub = MLPClassifier(hidden_layer_sizes=(64,32), max_iter=500, random_state=42)
mlp_sub.fit(X_q, y_q)
me_mlp_fidelity = accuracy_score(oracle(X_test), mlp_sub.predict(X_test))
me_mlp_acc      = accuracy_score(y_test, mlp_sub.predict(X_test))

print(f"ME Logistic    Fidelity: {me_lr_fidelity*100:.2f}%  Sub Acc: {me_lr_acc*100:.2f}%")
print(f"ME MLP         Fidelity: {me_mlp_fidelity*100:.2f}%  Sub Acc: {me_mlp_acc*100:.2f}%")
print(f"Oracle queries used: {query_count}")

In [ ]:
# Cell 7: EvasiveBench Report Card
print("=" * 65)
print(f"  {BENCH_VERSION} — REPORT CARD")
print(f"  Target: MLP(128,64,32) | Dataset: Wine | Date: {date.today()}")
print("=" * 65)

def risk(val, thresholds, labels):
    for t, l in zip(thresholds, labels):
        if val >= t:
            return l
    return labels[-1]

attacks = [
    {
        "name":    "Membership Inference — Loss-Based",
        "nist":    "NISTAML.033",
        "atlas":   "AML.T0024",
        "metric":  "AUC",
        "value":   mi_loss_auc,
        "display": f"{mi_loss_auc:.4f}",
        "risk":    risk(mi_loss_auc, [0.70, 0.60, 0.55], ["CRITICAL","HIGH","MODERATE","LOW"])
    },
    {
        "name":    "Membership Inference — Shadow Model",
        "nist":    "NISTAML.033",
        "atlas":   "AML.T0024",
        "metric":  "AUC",
        "value":   mi_shadow_auc,
        "display": f"{mi_shadow_auc:.4f}",
        "risk":    risk(mi_shadow_auc, [0.70, 0.60, 0.55], ["CRITICAL","HIGH","MODERATE","LOW"])
    },
    {
        "name":    "Model Extraction — Logistic Sub",
        "nist":    "NISTAML.031",
        "atlas":   "AML.T0030",
        "metric":  "Fidelity",
        "value":   me_lr_fidelity,
        "display": f"{me_lr_fidelity*100:.2f}%",
        "risk":    risk(me_lr_fidelity, [0.90, 0.75, 0.60], ["CRITICAL","HIGH","MODERATE","LOW"])
    },
    {
        "name":    "Model Extraction — MLP Sub",
        "nist":    "NISTAML.031",
        "atlas":   "AML.T0030",
        "metric":  "Fidelity",
        "value":   me_mlp_fidelity,
        "display": f"{me_mlp_fidelity*100:.2f}%",
        "risk":    risk(me_mlp_fidelity, [0.90, 0.75, 0.60], ["CRITICAL","HIGH","MODERATE","LOW"])
    },
]

print(f"\n{'Attack':<42} {'NIST':<15} {'Metric':<10} {'Value':<10} {'Risk'}")
print("-" * 95)
for a in attacks:
    print(f"{a['name']:<42} {a['nist']:<15} {a['metric']:<10} {a['display']:<10} {a['risk']}")

# Overall risk
critical = sum(1 for a in attacks if a["risk"] == "CRITICAL")
high     = sum(1 for a in attacks if a["risk"] == "HIGH")
overall  = "CRITICAL" if critical > 0 else "HIGH" if high > 0 else "MODERATE"

print()
print(f"Overall Risk : {overall}")
print(f"Critical     : {critical}/4 attacks")
print(f"High         : {high}/4 attacks")

print()
print("--- README ROW ---")
today = date.today().strftime("%Y-%m-%d")
print(f"| {today} | MLP(128,64,32)-wine | evasivebench-v0.1 | NISTAML.031/.033 | AML.T0024/T0030 | LLM06/LLM10 | MI AUC {mi_loss_auc:.4f} / ME Fidelity {me_lr_fidelity*100:.2f}% | Overall: {overall}. {critical}/4 attacks CRITICAL. |")

In [ ]:
# Cell 8: Export
export = {
    "benchmark":      BENCH_VERSION,
    "phase":          PHASE,
    "repo":           REPO,
    "run_timestamp":  datetime.now(timezone.utc).isoformat(),
    "target_model": {
        "architecture": "MLP(128,64,32)",
        "dataset":      "wine",
        "train_acc":    round(train_acc, 4),
        "test_acc":     round(test_acc, 4),
        "gen_gap":      round(gap, 4)
    },
    "results": {
        "membership_inference_loss_asr": round(mi_loss_asr, 4),
        "membership_inference_loss_auc": round(mi_loss_auc, 4),
        "membership_inference_shadow_asr": round(mi_shadow_asr, 4),
        "membership_inference_shadow_auc": round(mi_shadow_auc, 4),
        "model_extraction_lr_fidelity":  round(me_lr_fidelity, 4),
        "model_extraction_lr_acc":       round(me_lr_acc, 4),
        "model_extraction_mlp_fidelity": round(me_mlp_fidelity, 4),
        "model_extraction_mlp_acc":      round(me_mlp_acc, 4),
        "oracle_queries":                query_count
    }
}

ts    = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M")
fname = f"evasivebench_v01_{ts}.json"
with open(fname, "w") as f:
    json.dump(export, f, indent=2)
print(f"Saved: {fname}")

from google.colab import files
files.download(fname)

---
## Risk Scale

| Score | Risk |
|---|---|
| AUC 0.70+ / Fidelity 90%+ | CRITICAL |
| AUC 0.60+ / Fidelity 75%+ | HIGH |
| AUC 0.55+ / Fidelity 60%+ | MODERATE |
| Below threshold | LOW |

**Share Cell 7 output when done.**

**This is the final phase of the Evasive AI Lab v0.1 roadmap.**